# Træn YOLO11n — FCM Landmark Detection (v5_no_leak)

**Formål:** genskab projektets hovedmodel `v5_no_leak` (YOLO11n) på Colab T4 GPU i stedet for lokal Mac MPS.

**Hvorfor Colab:** Ultralytics' Task-Aligned-assigner har en kendt MPS-bug (`tal.py` shape-mismatch) der crasher træning undervejs — lokalt krævede det auto-resume for at gennemføre 80 epochs. Colab T4 (CUDA) rammer ikke bugen og er ~10× hurtigere end MPS.

**Input:** `composite_v1_no_leak_1280.tar.gz` (eller `.zip`) skal ligge i din Google Drive på stien `MyDrive/fcm_datasets/`. Datasættet er pre-resized til 1280px (max-dim) med bilinear interpolation — matcher Ultralytics' interne LetterBox-resize.

> **NDA:** dette er composite/syntetisk data — aldrig rigtige Videometer-billeder op til Colab.

**Output:** `models/v5_no_leak/best.pt` (downloades til din Mac sidst).

## Sådan kører du

1. Pre-resize datasættet lokalt (hvis ikke allerede gjort):
   ```bash
   .venv/bin/python scripts/02_yolo_pipeline/resize_for_colab.py \
       --src data/composite_v1_no_leak \
       --dst data/composite_v1_no_leak_1280
   ```
2. Pak op — vælg ét af:
   - `tar czf composite_v1_no_leak_1280.tar.gz data/composite_v1_no_leak_1280`
   - `zip -r composite_v1_no_leak_1280.zip data/composite_v1_no_leak_1280`
   (du har allerede `composite_v1_no_leak_1280.tar.gz` i projekt-roden)
3. Upload arkivet til Google Drive → `MyDrive/fcm_datasets/`
4. Åbn denne notebook i Colab
5. **Runtime → Change runtime type → T4 GPU**
6. **Runtime → Run all** (Cmd/Ctrl + F9)
7. Vent ~1-2 timer
8. Sidste celle downloader `best.pt` + `results.csv` til din Mac


## 1. Setup

Pinner samme Ultralytics-version som lokal installation så `best.pt` er kompatibel.

In [ ]:
!pip install ultralytics==8.4.49 -q

import ultralytics, torch
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch:     {torch.__version__}')
print(f'CUDA:        {torch.cuda.is_available()}')
print(f'GPU:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "INGEN — skift runtime til T4!"}')

## 2. Mount Google Drive

Drive-pop-up beder om autorisation første gang. Datasættet hentes derfra. Cellen leder efter både `.tar.gz` og `.zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

drive_folder = Path('/content/drive/MyDrive/fcm_datasets')
candidates = [
    drive_folder / 'composite_v1_no_leak_1280.tar.gz',
    drive_folder / 'composite_v1_no_leak_1280.zip',
]
archive = next((p for p in candidates if p.exists()), None)
assert archive is not None, (
    f'Mangler arkiv i {drive_folder}. Forventede en af: '
    f'{[p.name for p in candidates]}'
)
size_mb = archive.stat().st_size / 1024 / 1024
print(f'Arkiv fundet: {archive.name} ({size_mb:.1f} MB)')

## 3. Pak datasæt ud lokalt i Colab

Drive-I/O er langsom — kopiér til lokalt disk (`/content/`) før træning. Det er ~5-10× hurtigere under loader-fasen. Cellen håndterer både tar.gz og zip.

In [ ]:
import shutil, tarfile, time, zipfile
from pathlib import Path

dataset_root = Path('/content/data')
if dataset_root.exists():
    shutil.rmtree(dataset_root)
dataset_root.mkdir(parents=True)

start = time.time()
if archive.suffixes[-2:] == ['.tar', '.gz'] or archive.suffix == '.tgz':
    with tarfile.open(archive) as tf:
        tf.extractall(dataset_root)
elif archive.suffix == '.zip':
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(dataset_root)
else:
    raise ValueError(f'Ukendt arkiv-format: {archive.name}')
print(f'Udpakning tog {time.time() - start:.1f} sek')

# Find rod-mappen automatisk
candidates_yaml = list(dataset_root.rglob('data.yaml'))
assert candidates_yaml, 'Ingen data.yaml fundet i arkivet'
data_yaml_path = candidates_yaml[0]
dataset_dir = data_yaml_path.parent
print(f'Datasæt rod: {dataset_dir}')

# Verificer struktur
for split in ('train', 'val'):
    imgs = list((dataset_dir / 'images' / split).glob('*.png'))
    labels = list((dataset_dir / 'labels' / split).glob('*.txt'))
    print(f'  {split}: {len(imgs)} billeder, {len(labels)} labels')

## 4. Patch data.yaml til Colab-stier

Den oprindelige `data.yaml` har relative stier baseret på projekt-strukturen på din Mac. I Colab skal `path:` pege på den udpakkede mappe.

In [ ]:
import yaml

with open(data_yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = str(dataset_dir.resolve())
cfg.setdefault('train', 'images/train')
cfg.setdefault('val', 'images/val')

with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f)

print('Opdateret data.yaml:')
print(open(data_yaml_path).read())

## 5. Træn YOLO11n

Præcis samme opskrift som den lokale `v5_no_leak` (`03b_train_composite.py`):
- 80 epochs, imgsz=1280, patience=15
- Augmentations: flips, 180° rotation, mosaic, mixup, copy-paste

**Batch:** `8` (ikke 16). T4'eren OOM'er i `TaskAlignedAssigner` ved batch=16 fordi assigner-tensorerne skalerer med `batch × anchors × GT-per-billede`, og composites har mange frø pr. scene. Når den OOM'er falder Ultralytics tilbage til CPU-assigner → 10× langsommere. Rammer du stadig OOM ved 8, sæt `batch=4` (identisk med den lokale MPS-kørsel).


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data=str(data_yaml_path),
    epochs=80,
    imgsz=1280,
    batch=8,            # T4 OOM'er i TaskAlignedAssigner ved batch=16 (imgsz=1280 + mange frø/billede).
                        # batch=8 holder sig under 16 GB. Stadig OOM? → sæt 4 (= identisk med lokal v5).
    patience=15,
    project='runs',
    name='v5_no_leak',
    exist_ok=True,
    device=0,
    verbose=True,
    # Samme augmentations som 03b_train_composite.py (den lokale v5-opskrift)
    flipud=0.5,
    fliplr=0.5,
    degrees=180.0,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
)

print('\n=== TRÆNING FÆRDIG ===')
print(f'Precision: {results.box.mp:.4f}    (Jons krav: > 0.5)')
print(f'Recall:    {results.box.mr:.4f}    (Jons krav: > 0.3)')
print(f'mAP@50:    {results.box.map50:.4f}    (Jons krav: > 0.2)')
print(f'mAP@50-95: {results.box.map:.4f}    (bonus)')


## 6. Download best.pt + results.csv

Næste skridt på din Mac:

```bash
mkdir -p models/v5_no_leak
mv ~/Downloads/best.pt models/v5_no_leak/
mv ~/Downloads/results.csv models/v5_no_leak/
./switch_model.sh v5_no_leak
make demo                 # eller make upload-demo
```

`results.csv` er nødvendig for at footer-metrics i `upload_demo.py` virker for v5_no_leak.

In [ ]:
import glob
from pathlib import Path
from google.colab import files

# Find best.pt
weight_matches = glob.glob('**/v5_no_leak/weights/best.pt', recursive=True)
assert weight_matches, 'Ingen best.pt fundet — fejlede træningen?'
best_pt = weight_matches[0]

# Find results.csv (ligger i run-rod, ikke under weights/)
csv_matches = glob.glob('**/v5_no_leak/results.csv', recursive=True)
assert csv_matches, 'Ingen results.csv fundet'
results_csv = csv_matches[0]

print(f'best.pt:     {best_pt}  ({Path(best_pt).stat().st_size / 1024 / 1024:.2f} MB)')
print(f'results.csv: {results_csv}')

files.download(best_pt)
files.download(results_csv)